# Simulate the initial state

This code aims to simulate the initial state using SPGPE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import hbar, k as kB


# Physical constants
M_K39 = 39 * 1.6605e-27            # Mass of K39 in kg
A0 = 5.29177e-11                  # Bohr radius


# Dimensionless units
length_unit = 1e-6                                           # m
energy_unit = hbar**2 / (M_K39 * length_unit**2)             # J
time_unit   = hbar / energy_unit                             # s
temp_unit   = energy_unit / kB                               # K


# Simulation parameters 
L_phys     = 50e-6                # System size (m)
T_phys     = 100e-9               # Temperature (K)
n0_phys    = 20e12                # Density (m^-2)
omega_z    = 2 * np.pi * 1e3      # Trap frequency (Hz)
dt_phys    = 1e-6                 # Time step (s)
N_grid     = 256                  # Number of point
n_cut      = 2                    # Minimum occupation number (PGPE -> n_cut>>1)
save_every=100

# Initial state
a_ini   = 31 * A0                # Scattering length (m)
gamma_ini   = 0.01                # Bath coupling  
time_phys_thermalization  = 2 # Time evolutiom (s)                

file_database=r"D:\Users\Public\Documents\Louis\Simulations data\Data - relaxation\test" #File where simulations are saved, the following simulation will be saved in a labelled folder

### Do not touch (automatic)

In [ ]:
import json
from pathlib import Path
from dataclasses import dataclass


# Dimensionless & usefull quantities (DO NOT TOUCH)
L       = L_phys / length_unit
T       = np.round(T_phys / temp_unit, 4)
time    = time_phys_thermalization/time_unit
dt      = np.round(dt_phys/time_unit,5)
dx      = L / N_grid
n_steps = int(time/dt)
n0      = n0_phys * length_unit**2

osc_len_z  = np.sqrt(hbar / (M_K39 * omega_z))

g_phys_ini = (hbar**2 / M_K39) * np.sqrt(8 * np.pi) * (a_ini / osc_len_z)
g_ini    = np.round(g_phys_ini / (energy_unit * length_unit**2), 4)
print(g_ini)

mu_ini   = np.round(g_ini * n0, 4)

x    = np.linspace(-L/2, L/2, N_grid, endpoint=False)
k    = np.fft.fftfreq(N_grid, d=dx) * 2 * np.pi
KX, KY = np.meshgrid(k, k)
K2   = KX**2 + KY**2

E_cut= np.max([mu_ini]) + T * np.log(1+1/n_cut)
k_cut= np.sqrt(2*E_cut)
projector  = (np.sqrt(K2) <= k_cut).astype(complex)

T_BKT_ini= 2 * np.pi * n0 / (np.log(380 / g_ini))
lambda_db  = np.sqrt(2 * np.pi * hbar**2 / (M_K39 * T_phys * kB))
dk=2*np.pi/L
ek=dk**2/2
t_growth=1/(gamma_ini*max(mu_ini,T))

# Simulation stability

def check_stability(dx, dt, mu_ini):
        HEADER   = '\033[95m'
        OKGREEN  = '\033[92m'
        WARNING  = '\033[93m'
        FAIL     = '\033[91m'
        ENDC     = '\033[0m'
        BOLD     = '\033[1m'

        def report_line(label, value, limit, passed):
            status = f"{OKGREEN}[PASS]{ENDC}" if passed else f"{FAIL}[FAIL]{ENDC}"
            print(f"{label:<35} {value:>10.4f} | {limit:>10.4f} | {status}")

        print(f"\n{BOLD}{HEADER}" + "="*75 + f"{ENDC}")
        print(f"{BOLD}SIMULATION DIAGNOSTIC REPORT{ENDC}")
        print(f"{HEADER}" + "="*75 + f"{ENDC}")
        print(f"{BOLD}{'Metric':<35} {'Value':>10} | {'Limit':>10} | {'Status':<10}{ENDC}")
        print("-" * 75)

        kin_val  = dt / (dx**2);  kin_lim = 0.2 / np.pi
        pass_kin = kin_val < kin_lim
        report_line("Kinetic Phase Propagator (dt/dx²)", kin_val, kin_lim, pass_kin)

        nl_val  = mu_ini * dt;  nl_lim = 0.1
        pass_nl = nl_val < nl_lim
        report_line("Mean-field Phase Rotation (μ·dt)", nl_val, nl_lim, pass_nl)

        xi       = 1 / np.sqrt(2 * mu_ini)
        xi_res   = dx / xi;  xi_lim = 0.5
        pass_xi  = xi_res < xi_lim
        report_line("Grid Resolution (dx/ξ)", xi_res, xi_lim, pass_xi)

        print(f"{HEADER}" + "="*75 + f"{ENDC}")
        if not all([pass_kin, pass_nl, pass_xi]):
            print(f"{BOLD}{WARNING}RECOMMENDED SYSTEM TUNING:{ENDC}")
            if not pass_kin or not pass_nl:
                print("  * Temporal Accuracy: Reduce 'dt'.")
            if not pass_xi:
                print("  * Spatial Resolution: Increase 'N' or decrease 'L_grid'.")
            print(f"{HEADER}" + "="*75 + f"{ENDC}\n")
        else:
            print(f"{OKGREEN}{BOLD}System parameters are optimal.{ENDC}\n")

check_stability(dx, dt, mu_ini)


# Parameter recap and PGPE cutoff check
@dataclass
class SimParams:
    name: str
    value: float
    unit: str = ""
    fmt: str = ".4f"

# Collect all data into a structured list
data = [
    SimParams("L", L),
    SimParams("T", T),
    SimParams("Time (thermal)", time_phys_thermalization/time_unit),
    SimParams("T_growth", t_growth),
    SimParams("dt", dt, fmt=".6f"),
    SimParams("n_steps", n_steps, fmt=".2e"),
    SimParams("n0", n0, fmt=".6f"),
    SimParams("mu_ini", mu_ini, fmt=".6f"),
    SimParams("E_cut", E_cut, fmt=".6f"),
    SimParams("k_cut", k_cut, fmt=".6f"),
    SimParams("T_BKT_ini", T_BKT_ini, fmt=".6f"),
]

# Print neatly
print(f"\n{'='*45}\n{'SIMULATION PARAMETER RECAP':^45}\n{'='*45}")
for p in data:
    print(f"{p.name:<20} : {p.value:>{p.fmt}} {p.unit}")

# Logic checks
print(f"\n{'='*45}\n{'CUTOFF VALIDATION':^45}\n{'='*45}")
print(f"Modes: {int(np.sum(projector != 0)):<8} / {N_grid**2}")
print(f"E_cut vs Max(mu): {'OK' if E_cut > mu_ini else 'WARNING'}")
print(f"k_cut vs k_max:   {'OK' if k_cut < np.sqrt(K2).max() else 'WARNING'}")



# Folder creation
base_dir = Path(file_database)
folder_name = f"T={T:.2f}-mu_ini={mu_ini:.2f}-n0={n0:.2f}-gamma={gamma_ini:.4f}-dt={dt}"
full_path = base_dir / folder_name
full_path.mkdir(parents=True, exist_ok=True)

parameters = {
    "L_phys": L_phys,
    "T_phys": T_phys,
    "n0_phys": n0_phys,
    "omega_z": omega_z,
    "mu_ini": mu_ini,
    "L": L,
    "T": T,
    "gamma_ini": gamma_ini,
    "dt": dt,
    "n_steps": n_steps,
    "N_grid": N_grid,
    "k_cut": k_cut,
    "save_every":save_every
}


with open(full_path / "metadata_initial_state.json", "w") as f:
    json.dump(parameters, f, indent=4)

print(f"\n--- SAVING ---")
print(f"Metadata successfully saved to: '{full_path / 'metadata_initial_state.json'}'")




# Solver

In [ ]:
def evolve_SPGPE(psi, g, mu, gamma):
    psi_k = np.fft.fft2(psi) * projector

    V = g * np.abs(psi)**2
    force = (-1j - gamma) * (V - mu) * psi

    noise = (np.random.normal(size=psi.shape)
                + 1j*np.random.normal(size=psi.shape)) / np.sqrt(2)

    noise_scale = np.sqrt(2 * gamma * T * dt / dx**2)
    dW = np.fft.fft2(noise_scale * noise) * projector

    drift = (-1j - gamma) * (0.5 * K2)
    psi_k = (psi_k + np.fft.fft2(force)*projector*dt + dW) / (1 - drift*dt)

    return np.fft.ifft2(psi_k * projector)

# Initial state thermalization

The following line allows to load an initial state and continue a simualtion by running it longer afterward. You don't need it if it's a new simulation

In [ ]:
psi_storage = np.memmap(full_path/'psi_gound_state.dat',
                        dtype=np.complex128,
                        mode='r+',
                        shape=(n_steps//save_every+1,N_grid,N_grid))
psi0=psi_storage[-1]
del psi_storage

The following line run the simulation and save write it in a .dat file while running every $10\times$ save_every.

In [ ]:
import tqdm


file=full_path/'psi_gound_state.dat'
# file.unlink()  #     Deletes the existing file

psi_storage = np.memmap(file,
                        dtype=np.complex128,
                        mode='w+',
                        shape=(n_steps//save_every+1,N_grid,N_grid))


psi= np.sqrt(n0)*np.ones((N_grid,N_grid), dtype=np.complex128)
psi*=np.exp(1j*2*np.pi*np.random.rand(N_grid,N_grid))
psi+=0.01*(np.random.rand(N_grid,N_grid)+1j*np.random.rand(N_grid,N_grid))

psi = (np.random.normal(size=(N_grid, N_grid)) + 
       1j * np.random.normal(size=(N_grid, N_grid))) * .1

# psi=psi0
j=0
for i in tqdm.tqdm(range(n_steps)):
    
    psi= evolve_SPGPE(psi, g_ini, mu_ini, gamma_ini)

    if i % save_every ==0:
        psi_storage[j,:,:] = psi

        if j%10 ==0:
            psi_storage.flush()

        j+=1

psi_storage.flush()



# Sanity checks

Load the field

In [ ]:

path_ini=r"YourFolder\psi_gound_state.dat"

psi_storage = np.memmap(path_ini,
                        dtype=np.complex128,
                        mode='r',
                        shape=(n_steps//save_every+1,N_grid,N_grid))

Ns=[] # Mean density over time
Psi_s=[] # Field over time
for i in range(n_steps//save_every):
    psi_t= psi_storage[i]
    N=np.mean(np.abs(psi_t)**2)
    Ns.append(N)
    Psi_s.append(psi_t)


In [ ]:
plt.figure()
plt.imshow(np.abs(psi_storage[-1])**2, vmin=0)
plt.colorbar()
plt.figure()
plt.imshow(np.angle(psi_storage[-1]))
plt.colorbar()


# Sanity checks

#### N convergence

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",          
    "font.size": 10,                  
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "xtick.direction": "in",            
    "ytick.direction": "in",
    "xtick.top": True,                   
    "ytick.right": True,
    "xtick.major.size": 4,
    "xtick.minor.size": 2,
    "ytick.major.size": 4,
    "ytick.minor.size": 2,
    "axes.grid": False,                 
})

fig, ax = plt.subplots(figsize=(3.375/1.5, 2.5/1.5))

time_steps = np.arange(len(Ns)) * dt * time_unit * save_every
ax.plot(time_steps, Ns, linewidth=1.2, color='black', label='Density')

ax.set_xlabel(r'Time $t$ (s)')
ax.set_ylabel(r'$n=\langle|\psi|^2\rangle$')
ax.set_xscale('log')
# ax.set_xlim(time_steps[0], time_steps[-1])

plt.tight_layout(pad=0.5)
plt.show()

### 'Condensate' fraction $k_0$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

psi_storage = np.memmap(full_path/'psi_gound_state.dat',
                        dtype=np.complex128,
                        mode='r',
                        shape=(n_steps//save_every+1,N_grid,N_grid))

condensate_fractions = []
for i in range(n_steps//save_every):
    psi_t = psi_storage[i]
    
    psi_k = np.fft.fft2(psi_t)
    
    k0_occupation = np.abs(psi_k[0, 0])**2
    
    N_total = np.sum(np.abs(psi_k)**2)
    
    condensate_fraction = k0_occupation / N_total
    condensate_fractions.append(condensate_fraction)


plt.rcParams.update(plt.rcParamsDefault)


fig, ax = plt.subplots(constrained_layout=True, figsize=(3.4, 2.5))
time=np.arange(0,(n_steps//save_every)*dt,dt)
ax.plot(time,condensate_fractions, color='k', linestyle='-', label=r'$n_0 / N$')


ax.set_xlabel(r'Time ($t$)')
ax.set_ylabel(r'Condensate fraction $f_0$')
ax.grid(True, linestyle=':', alpha=0.6, zorder=0)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)


plt.show()

### $g^{(1)}(r)$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def compute_g1(Psi_array):
    psi_k = np.fft.fft2(Psi_array, axes=(-2, -1))
    corr = np.fft.ifft2(np.abs(psi_k)**2, axes=(-2, -1))
    corr = np.fft.fftshift(corr, axes=(-2, -1))
    g1_sum = np.mean(corr.real, axis=0)
    g1_sum /= np.max(g1_sum)
    return g1_sum

g1_2d = compute_g1(np.array(Psi_s[-200:]))

center = N_grid // 2
g1_1d = g1_2d[center, center:]
x = np.linspace(0, L/2, N_grid // 2)

def power_law(x, A, eta):
    return A * x**(-eta)

def exponential(x, A, xi):
    return A * np.exp(-x / xi)

fit_mask = x > 0
x_fit = x[fit_mask]
y_fit = g1_1d[fit_mask]

popt_pl, _ = curve_fit(power_law, x_fit, y_fit, p0=[0.9, 0.13])
A_pl, eta_pl = popt_pl

popt_exp, _ = curve_fit(exponential, x_fit, y_fit, p0=[1.0, L/4])
A_exp, xi_exp = popt_exp

print(f"Power Law: eta = {eta_pl:.4f}")
print(f"Exponential: correlation length (xi) = {xi_exp:.4f}")

fig, ax = plt.subplots(figsize=(3.375/1.5, 2.5/1.5))


ax.plot(x, g1_1d, 'o', label='Data', markersize=3.5, 
        markerfacecolor='none', markeredgecolor='gray', markeredgewidth=0.8)

ax.plot(x_fit, power_law(x_fit, *popt_pl), '-.', 
        label=rf'$r^{{-\eta}}$ ($\eta={eta_pl:.3f}$)', color='black', linewidth=1)

ax.plot(x_fit, exponential(x_fit, *popt_exp), '--', 
        label=rf"$e^{{-x/\xi}}$ ($\xi={xi_exp:.2f}$)", color='tab:red', linewidth=1.2)

ax.set_xscale('log')
# ax.set_yscale('log')

ax.set_xlim(x[1], L/2)
ax.set_ylim(1e-6,1.05)

ax.set_xlabel(r"Distance $r$")
ax.set_ylabel(r"$g^{(1)}(r)$")

leg = ax.legend(frameon=False, fontsize=8, loc='best')

plt.tight_layout(pad=0.5)

plt.show()